In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall
from sklearn.utils.class_weight import compute_class_weight
import tensorflow.keras.backend as K


import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-05-20 11:16:33.450409: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779275793.859250      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779275793.963650      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779275794.956895      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779275794.956959      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779275794.956962      57 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/traintestval-lstm"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train (1).npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train (1).npy'))
X_val = np.load(os.path.join(INPUT_PATH, 'X_val (1).npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val (1).npy'))
X_test = np.load(os.path.join(INPUT_PATH, 'X_test (1).npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test (1).npy'))

In [4]:
X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train must be 3D, got {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val must be 3D, got {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test must be 3D, got {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train and y_train have different sample counts"
assert len(X_val) == len(y_val), "X_val and y_val have different sample counts"
assert len(X_test) == len(y_test), "X_test and y_test have different sample counts"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (190797, 10, 133) float32
y_val:   (190797,) int32
X_test:  (238447, 10, 133) float32
y_test:  (238447,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [187169   3628]
Test class distribution: [234296   4151]


In [5]:
import sys
MODULE_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-utils"
if MODULE_PATH not in sys.path:
    sys.path.append(MODULE_PATH)

from model_utils import (
    create_bilstm,
    get_callbacks,
    find_best_threshold,
    full_evaluation,
)

In [6]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}
print("Train class weights:", class_weights_dict)

lstm_units_options = [32, 64, 80]
dropout_rate_options = [0.3, 0.4, 0.5]
batch_size_options = [256, 512]
num_epochs = 15

results = []

for n_units in lstm_units_options:
    for dropout_rate in dropout_rate_options:
        for batch_size in batch_size_options:
            print(f"\nTesting: units={n_units}, dropout={dropout_rate}, batch_size={batch_size}")

            K.clear_session()

            model = create_bilstm(
                n_units=n_units,
                dropout=dropout_rate,
                seq_len=X_train.shape[1],
                n_features=X_train.shape[2],
                lr=1e-3
            )

            history = model.fit(
                X_train, y_train,
                epochs=num_epochs,
                batch_size=batch_size,
                class_weight=class_weights_dict,
                validation_data=(X_val, y_val),
                shuffle=True,
                verbose=0
            )

            val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision = model.evaluate(
                X_val, y_val, verbose=0
            )

            print(
                f"Val AUROC: {val_auroc:.4f} | "
                f"Val AUPRC: {val_auprc:.4f} | "
                f"Recall: {val_recall:.4f} | "
                f"Precision: {val_precision:.4f}"
            )

            results.append((
                n_units, dropout_rate, batch_size,
                val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision
            ))

# Select by validation AUPRC rather than AUROC
best_result = sorted(results, key=lambda x: x[6], reverse=True)[0]

print("\n" + "="*60)
print("Best Hyperparameters:")
print(f"- LSTM Units:   {best_result[0]}")
print(f"- Dropout Rate: {best_result[1]}")
print(f"- Batch Size:   {best_result[2]}")
print(f"- Val AUROC:    {best_result[5]:.4f}")
print(f"- Val AUPRC:    {best_result[6]:.4f}")
print("="*60)

Train class weights: {0: 0.5407133742841034, 1: 6.64048833819242}

Testing: units=32, dropout=0.3, batch_size=256


I0000 00:00:1779275867.784437      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779275867.790577      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779275877.358513     131 cuda_dnn.cc:529] Loaded cuDNN version 91002


Val AUROC: 0.7278 | Val AUPRC: 0.0673 | Recall: 0.3950 | Precision: 0.0859

Testing: units=32, dropout=0.3, batch_size=512
Val AUROC: 0.7514 | Val AUPRC: 0.0714 | Recall: 0.5063 | Precision: 0.0770

Testing: units=32, dropout=0.4, batch_size=256
Val AUROC: 0.7393 | Val AUPRC: 0.0548 | Recall: 0.4824 | Precision: 0.0736

Testing: units=32, dropout=0.4, batch_size=512
Val AUROC: 0.7389 | Val AUPRC: 0.0747 | Recall: 0.5025 | Precision: 0.0731

Testing: units=32, dropout=0.5, batch_size=256
Val AUROC: 0.7340 | Val AUPRC: 0.0609 | Recall: 0.4413 | Precision: 0.0824

Testing: units=32, dropout=0.5, batch_size=512
Val AUROC: 0.7541 | Val AUPRC: 0.0587 | Recall: 0.5502 | Precision: 0.0700

Testing: units=64, dropout=0.3, batch_size=256
Val AUROC: 0.6862 | Val AUPRC: 0.0554 | Recall: 0.3280 | Precision: 0.0813

Testing: units=64, dropout=0.3, batch_size=512
Val AUROC: 0.7237 | Val AUPRC: 0.0573 | Recall: 0.3564 | Precision: 0.0797

Testing: units=64, dropout=0.4, batch_size=256
Val AUROC: 0.715

In [7]:
K.clear_session()

final_model = create_bilstm(
    n_units=best_result[0],
    dropout=best_result[1],
    seq_len=X_train.shape[1],
    n_features=X_train.shape[2],
    lr=1e-3
)

callbacks = get_callbacks(
    checkpoint_path='best_lstm_model.keras',
    monitor='val_auprc'
)

history = final_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=best_result[2],
    class_weight=class_weights_dict,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/50
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6461 - auprc: 0.2465 - auroc: 0.7716 - loss: 0.5851 - precision: 0.1511 - recall: 0.7687
Epoch 1: val_auprc improved from -inf to 0.10579, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.6464 - auprc: 0.2467 - auroc: 0.7718 - loss: 0.5849 - precision: 0.1512 - recall: 0.7688 - val_accuracy: 0.7728 - val_auprc: 0.1058 - val_auroc: 0.8202 - val_loss: 0.4360 - val_precision: 0.0570 - val_recall: 0.7042 - learning_rate: 0.0010
Epoch 2/50
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7964 - auprc: 0.3997 - auroc: 0.8927 - loss: 0.4098 - precision: 0.2460 - recall: 0.8501
Epoch 2: val_auprc did not improve from 0.10579
285/285 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.7964 - auprc: 0.3998 - auroc: 0.8927 - loss: 0.4097 - precision: 0.2460 - recall: 0.8501 - val_accuracy: 0.8198 - val_auprc: 0.0942 - val_auroc: 0.8179 - val_loss: 0.3797 - val_precision: 0.0650 - va

In [8]:
val_prob = final_model.predict(X_val, verbose=0).ravel()

best_val_threshold = find_best_threshold(
    y_true=y_val,
    y_prob=val_prob,
    min_sensitivity=0.80,
    threshold_range=(0.05, 0.95),
    step=0.01
)

print("Best threshold from validation:")
print(best_val_threshold)

Best threshold from validation:
{'threshold': 0.39, 'accuracy': 0.6817717259705341, 'sensitivity': np.float64(0.8029217199558986), 'specificity': np.float64(0.6794234087909857), 'precision': 0.046300564253357705, 'recall': 0.8029217199558986, 'f1': 0.08755240972003066, 'youden_j': np.float64(0.4823451287468843), 'tn': 127167, 'fp': 60002, 'fn': 715, 'tp': 2913}


In [9]:
test_prob = final_model.predict(X_test, verbose=0).ravel()

full_evaluation(
    y_true=y_test,
    y_prob=test_prob,
    threshold=best_val_threshold["threshold"],
    label="TEST"
)

df_lstm = pd.DataFrame({
    'y_true': y_test,
    'pred_lstm': test_prob,
    'pred_label': (test_prob >= best_val_threshold["threshold"]).astype(int)
})


  TEST — threshold = 0.39
  AUROC        : 0.8400
  AUPRC        : 0.1093
  Accuracy     : 0.6876
  Sensitivity  : 0.8285
  Specificity  : 0.6851
  Precision    : 0.0445
  Recall       : 0.8285
  F1-score     : 0.0845
  Youden's J   : 0.5136
  Confusion    : TN=160525 FP=73771 FN=712 TP=3439

